# 03 — Semantic Model Training (Yuno-Style)

Two-stage training of a custom anime sentence transformer:
1. **MLM Pre-Training** — Domain-adaptive masked language modeling on anime text
2. **Triplet Loss Fine-Tuning** — Learn that reviews of the same anime embed close together

**Base model**: `sentence-transformers/all-MiniLM-L6-v2` (384-dim, 22M params)

**Hardware**: T4 GPU (~16GB VRAM) — Kaggle or Colab free tier

**Requires**: Run `02_preprocessing.ipynb` first (needs `corpus.jsonl` and `triplets.jsonl`).

In [1]:
import sys, torch
print("Python:", sys.executable)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

Python: C:\Users\jeddh\Projects\animetracker\notebooks\.venv311\Scripts\python.exe
Torch: 2.5.1+cu121
Torch CUDA: 12.1
CUDA available: True
GPU count: 1


In [2]:
# Install dependencies (run once per session)
!pip install -q "sentence-transformers==5.2.2" "transformers>=4.41,<5.0" "accelerate>=0.20.3,<2" torch datasets tqdm

In [3]:
import json
import torch
import numpy as np
from tqdm.auto import tqdm
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForMaskedLM,
    DataCollatorForLanguageModeling,
    TrainingArguments,
    Trainer,
    TrainerCallback
)
from sentence_transformers import SentenceTransformer, InputExample, losses
from sentence_transformers.evaluation import TripletEvaluator

DATA_DIR = Path("data")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram_bytes = getattr(props, "total_memory", None)
    if vram_bytes is not None:
        print(f"VRAM: {vram_bytes / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
VRAM: 8.6 GB


## Stage 1: MLM Pre-Training (Domain Adaptation)

Teach the model anime-specific vocabulary by masked language modeling on our corpus.
This follows Yuno's approach: before training for similarity, adapt the language model to the anime domain.

In [4]:
# Load corpus texts
corpus_texts = []
with open(DATA_DIR / "corpus.jsonl", "r") as f:
    for line in f:
        entry = json.loads(line)
        corpus_texts.append(entry["text"])

print(f"Corpus size: {len(corpus_texts):,} documents")
print(f"Total chars: {sum(len(t) for t in corpus_texts):,}")

Corpus size: 5,000 documents
Total chars: 21,211,786


In [5]:
BASE_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL)

print(f"Base model: {BASE_MODEL}")
print(f"Parameters: {sum(p.numel() for p in mlm_model.parameters()):,}")

Some weights of BertForMaskedLM were not initialized from the model checkpoint at sentence-transformers/all-MiniLM-L6-v2 and are newly initialized: ['cls.predictions.bias', 'cls.predictions.decoder.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Base model: sentence-transformers/all-MiniLM-L6-v2
Parameters: 22,744,506


In [6]:
# Tokenize corpus for MLM
MAX_SEQ_LEN = 256

# Import from a real module so Windows multiprocessing workers can resolve the class.
import sys
from pathlib import Path
if not Path("training_datasets.py").exists() and Path("notebooks/training_datasets.py").exists():
    sys.path.append(str(Path("notebooks").resolve()))
from training_datasets import AnimeCorpusDataset

# Split long documents into chunks of max_length
chunked_texts = []
for text in tqdm(corpus_texts, desc="Chunking corpus"):
    # Split text into chunks that fit within max token length
    words = text.split()
    chunk = []
    chunk_len = 0
    for word in words:
        chunk.append(word)
        chunk_len += len(word.split()) + 1  # Rough token estimate
        if chunk_len >= 200:  # Leave headroom for special tokens
            chunked_texts.append(" ".join(chunk))
            chunk = []
            chunk_len = 0
    if chunk:
        chunked_texts.append(" ".join(chunk))

print(f"Chunked texts: {len(chunked_texts):,} (from {len(corpus_texts):,} documents)")

mlm_dataset = AnimeCorpusDataset(chunked_texts, tokenizer, MAX_SEQ_LEN)
print(f"MLM dataset size: {len(mlm_dataset):,}")

Chunking corpus:   0%|          | 0/5000 [00:00<?, ?it/s]

Chunked texts: 38,593 (from 5,000 documents)
MLM dataset size: 38,593


In [7]:
# MLM Data Collator: 15% masking (80% [MASK], 10% random, 10% unchanged)
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# Training arguments
MLM_EPOCHS = 5
MLM_BATCH_SIZE = 32
MLM_LR = 1e-5

mlm_output_dir = MODEL_DIR / "mlm_pretrained"

training_args = TrainingArguments(
    output_dir=str(mlm_output_dir),
    num_train_epochs=MLM_EPOCHS,
    per_device_train_batch_size=MLM_BATCH_SIZE,
    learning_rate=MLM_LR,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_strategy="steps",
    logging_steps=25,
    logging_first_step=True,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    dataloader_pin_memory=torch.cuda.is_available(),
    dataloader_num_workers=2,
    disable_tqdm=True,
    report_to="none"
)

print(f"MLM Training: {MLM_EPOCHS} epochs, batch_size={MLM_BATCH_SIZE}, lr={MLM_LR}")
print(f"Output: {mlm_output_dir}")

MLM Training: 5 epochs, batch_size=32, lr=1e-05
Output: models\mlm_pretrained


In [8]:
# Train MLM
class NotebookProgressCallback(TrainerCallback):
    """Compact step/epoch progress bar for notebook runs."""
    def __init__(self):
        self.pbar = None

    def on_train_begin(self, args, state, control, **kwargs):
        total_steps = state.max_steps if state.max_steps and state.max_steps > 0 else None
        self.pbar = tqdm(total=total_steps, desc="MLM training", dynamic_ncols=True)

    def on_step_end(self, args, state, control, **kwargs):
        if self.pbar is not None:
            self.pbar.n = int(state.global_step)
            self.pbar.refresh()

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.pbar is None or not logs:
            return
        postfix = {}
        if "loss" in logs:
            postfix["loss"] = f"{logs['loss']:.4f}"
        if "learning_rate" in logs:
            postfix["lr"] = f"{logs['learning_rate']:.2e}"
        if "epoch" in logs:
            postfix["epoch"] = f"{logs['epoch']:.2f}"
        if postfix:
            self.pbar.set_postfix(postfix)

    def on_train_end(self, args, state, control, **kwargs):
        if self.pbar is not None:
            self.pbar.close()
            self.pbar = None

trainer = Trainer(
    model=mlm_model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=mlm_dataset,
    callbacks=[NotebookProgressCallback()]
)

print("Starting MLM pre-training...")
trainer.train()

# Save the MLM-pretrained model
trainer.save_model(str(mlm_output_dir / "final"))
tokenizer.save_pretrained(str(mlm_output_dir / "final"))
print(f"\nMLM pre-training complete. Saved to {mlm_output_dir / 'final'}")

Starting MLM pre-training...


MLM training:   0%|                                                                             | 0/6035 [00:0…

{'loss': 10.9385, 'grad_norm': inf, 'learning_rate': 0.0, 'epoch': 0.0008285004142502071}
{'loss': 10.856, 'grad_norm': 24.82681655883789, 'learning_rate': 3.80794701986755e-07, 'epoch': 0.020712510356255178}
{'loss': 10.7198, 'grad_norm': 24.882211685180664, 'learning_rate': 7.94701986754967e-07, 'epoch': 0.041425020712510356}
{'loss': 10.3942, 'grad_norm': 25.050369262695312, 'learning_rate': 1.208609271523179e-06, 'epoch': 0.06213753106876554}
{'loss': 9.8925, 'grad_norm': 27.083656311035156, 'learning_rate': 1.6059602649006622e-06, 'epoch': 0.08285004142502071}
{'loss': 9.2016, 'grad_norm': 21.74829864501953, 'learning_rate': 2.0198675496688742e-06, 'epoch': 0.1035625517812759}
{'loss': 8.4628, 'grad_norm': 9.446969985961914, 'learning_rate': 2.4337748344370862e-06, 'epoch': 0.12427506213753108}
{'loss': 7.9321, 'grad_norm': 6.40696382522583, 'learning_rate': 2.8476821192052982e-06, 'epoch': 0.14498757249378624}
{'loss': 7.6223, 'grad_norm': 4.963611125946045, 'learning_rate': 3.26

## Stage 2: Triplet Loss Fine-Tuning

Now fine-tune the MLM-pretrained model with triplet loss so that:
- Reviews of the **same anime** embed close together
- Reviews of **different anime** embed far apart

This is the core training that makes the model understand anime similarity.

In [9]:
# Load triplets
triplets_data = []
with open(DATA_DIR / "triplets.jsonl", "r") as f:
    for line in f:
        triplets_data.append(json.loads(line))

print(f"Total triplets: {len(triplets_data):,}")

# Split into train/eval (95%/5%)
np.random.shuffle(triplets_data)
split_idx = int(0.95 * len(triplets_data))
train_triplets = triplets_data[:split_idx]
eval_triplets = triplets_data[split_idx:]

print(f"Train: {len(train_triplets):,}, Eval: {len(eval_triplets):,}")

Total triplets: 14,954
Train: 14,206, Eval: 748


In [10]:
# Build SentenceTransformer from MLM-pretrained weights
# Load the base architecture but swap in our MLM-pretrained weights
from sentence_transformers import models

mlm_pretrained_path = str(mlm_output_dir / "final")

# Word embedding model (transformer)
word_embedding = models.Transformer(mlm_pretrained_path, max_seq_length=MAX_SEQ_LEN)

# Mean pooling layer
pooling = models.Pooling(
    word_embedding.get_word_embedding_dimension(),
    pooling_mode_mean_tokens=True
)

# Build the sentence transformer
model = SentenceTransformer(modules=[word_embedding, pooling], device=str(device))
print(f"Model embedding dimension: {model.get_sentence_embedding_dimension()}")

Some weights of BertModel were not initialized from the model checkpoint at models\mlm_pretrained\final and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model embedding dimension: 384


In [11]:
# Convert triplets to InputExamples
train_examples = [
    InputExample(texts=[t["anchor"], t["positive"], t["negative"]])
    for t in train_triplets
]

# DataLoader
TRIPLET_BATCH_SIZE = 16
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=TRIPLET_BATCH_SIZE)

# Triplet loss
TRIPLET_MARGIN = 0.3
triplet_loss = losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=TRIPLET_MARGIN
)

# Evaluator
eval_anchors = [t["anchor"] for t in eval_triplets]
eval_positives = [t["positive"] for t in eval_triplets]
eval_negatives = [t["negative"] for t in eval_triplets]

evaluator = TripletEvaluator(
    anchors=eval_anchors,
    positives=eval_positives,
    negatives=eval_negatives,
    name="anime-triplet-eval"
)

print(f"Train examples: {len(train_examples):,}")
print(f"Batch size: {TRIPLET_BATCH_SIZE}")
print(f"Triplet margin: {TRIPLET_MARGIN}")
print(f"Steps per epoch: {len(train_dataloader):,}")

Train examples: 14,206
Batch size: 16
Triplet margin: 0.3
Steps per epoch: 888


In [12]:
# Training hyperparameters (conservative to reduce semantic drift)
TRIPLET_EPOCHS = 4
TRIPLET_LR = 1e-5
WARMUP_STEPS = int(0.05 * len(train_dataloader) * TRIPLET_EPOCHS)

triplet_output_dir = str(MODEL_DIR / "anime_semantic")

print(f"Triplet training: {TRIPLET_EPOCHS} epochs, lr={TRIPLET_LR}")
print(f"Warmup steps: {WARMUP_STEPS}")
print(f"Output: {triplet_output_dir}")

Triplet training: 4 epochs, lr=1e-05
Warmup steps: 177
Output: models\anime_semantic


In [13]:
# Train!
model.fit(
    train_objectives=[(train_dataloader, triplet_loss)],
    evaluator=evaluator,
    epochs=TRIPLET_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    optimizer_params={"lr": TRIPLET_LR},
    weight_decay=0.01,
    output_path=triplet_output_dir,
    evaluation_steps=max(100, len(train_dataloader) // 2),
    save_best_model=True,
    show_progress_bar=True
)

print(f"\nTriplet fine-tuning complete!")
print(f"Best model saved to: {triplet_output_dir}")

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Step,Training Loss,Validation Loss,Anime-triplet-eval Cosine Accuracy
444,No log,No log,0.902406
888,0.167400,No log,0.914438
1332,0.077100,No log,0.929144
1776,0.061400,No log,0.933155
2220,0.051600,No log,0.938503
2664,0.047100,No log,0.941176
3108,0.040900,No log,0.942513
3552,0.038700,No log,0.943850



Triplet fine-tuning complete!
Best model saved to: models\anime_semantic


## Quick Validation

Test the model with some anime-specific queries to verify it learned domain semantics.

In [14]:
# Load the best model
trained_model = SentenceTransformer(triplet_output_dir, device=str(device))

# Test queries
test_queries = [
    "cute girls doing cute things in high school",
    "dark psychological thriller with mind games",
    "mecha robots fighting in space",
    "romantic comedy with tsundere characters",
    "isekai protagonist transported to fantasy world",
    "sports anime about basketball"
]

# Some anime descriptions to compare against
test_anime = [
    "K-On! is about a group of high school girls who form a light music club. They spend most of their time drinking tea and eating snacks rather than practicing, creating a warm slice of life comedy.",
    "Death Note follows a genius high school student who finds a supernatural notebook that kills anyone whose name is written in it. A psychological cat-and-mouse game ensues.",
    "Neon Genesis Evangelion features giant mecha called EVAs piloted by teenagers to fight mysterious beings called Angels threatening humanity.",
    "Toradora is a romantic comedy about a fierce girl and a gentle boy who help each other pursue their respective crushes, only to develop feelings for each other.",
    "Re:Zero follows Subaru who is transported to a fantasy world where he discovers he has the ability to return from death, using this power to save those he cares about.",
    "Slam Dunk follows delinquent Hanamichi Sakuragi as he joins the basketball team to impress a girl but gradually falls in love with the sport itself."
]

test_anime_names = ["K-On!", "Death Note", "Evangelion", "Toradora", "Re:Zero", "Slam Dunk"]

# Encode
query_embeddings = trained_model.encode(test_queries, normalize_embeddings=True)
anime_embeddings = trained_model.encode(test_anime, normalize_embeddings=True)

# Compute cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

sim_matrix = cosine_similarity(query_embeddings, anime_embeddings)

print("Query → Best Match (cosine similarity):")
print("=" * 60)
for i, query in enumerate(test_queries):
    best_idx = np.argmax(sim_matrix[i])
    best_score = sim_matrix[i][best_idx]
    print(f"  '{query}'")
    print(f"    → {test_anime_names[best_idx]} ({best_score:.3f})")
    # Show all scores
    ranked = sorted(enumerate(sim_matrix[i]), key=lambda x: x[1], reverse=True)
    for idx, score in ranked:
        marker = " ✓" if idx == best_idx else ""
        print(f"      {test_anime_names[idx]}: {score:.3f}{marker}")
    print()

Query → Best Match (cosine similarity):
  'cute girls doing cute things in high school'
    → K-On! (0.347)
      K-On!: 0.347 ✓
      Toradora: 0.345
      Slam Dunk: 0.261
      Evangelion: -0.014
      Death Note: -0.119
      Re:Zero: -0.298

  'dark psychological thriller with mind games'
    → Death Note (0.474)
      Death Note: 0.474 ✓
      Evangelion: 0.202
      Re:Zero: 0.145
      Slam Dunk: -0.037
      Toradora: -0.039
      K-On!: -0.094

  'mecha robots fighting in space'
    → Evangelion (0.416)
      Evangelion: 0.416 ✓
      Re:Zero: 0.144
      Slam Dunk: 0.053
      Death Note: 0.032
      K-On!: -0.006
      Toradora: -0.139

  'romantic comedy with tsundere characters'
    → Toradora (0.457)
      Toradora: 0.457 ✓
      K-On!: 0.208
      Slam Dunk: 0.185
      Death Note: -0.138
      Evangelion: -0.157
      Re:Zero: -0.240

  'isekai protagonist transported to fantasy world'
    → Re:Zero (0.275)
      Re:Zero: 0.275 ✓
      Death Note: 0.125
      Evangelio

In [15]:
import json
import re
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer

# Load trained model
trained_model = SentenceTransformer(triplet_output_dir, device=str(device))

# Load corpus entries
entries = []
with open(DATA_DIR / "corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        x = json.loads(line)
        title = (x.get("title") or "").strip()
        text = (x.get("text") or "").strip()
        if not title or not text:
            continue

        # Parse genres from "Genres: ..." line in corpus text
        m = re.search(r"^Genres:\s*(.+)$", text, flags=re.MULTILINE)
        genres = []
        if m:
            genres = [g.strip() for g in m.group(1).split(",") if g.strip()]

        # Keep only first part to reduce noise from very long reviews
        short_text = text[:1400]
        entries.append({"title": title, "text": short_text, "genres": genres})

# Dedupe by title
dedup = {}
for e in entries:
    dedup[e["title"].lower()] = e
entries = list(dedup.values())

# Build a diverse eval pool
rng = np.random.default_rng(42)
genre_targets = [
    "Action", "Adventure", "Comedy", "Romance", "Slice of Life",
    "Psychological", "Thriller", "Mystery", "Sci-Fi", "Mecha",
    "Fantasy", "Supernatural", "Sports", "Drama", "Horror"
]

by_genre = defaultdict(list)
for e in entries:
    for g in e["genres"]:
        by_genre[g].append(e)

pool = []
seen = set()

def add_entry(e):
    key = e["title"].lower()
    if key not in seen:
        seen.add(key)
        pool.append(e)

# sample up to 70 per target genre
for g in genre_targets:
    candidates = by_genre.get(g, [])
    if not candidates:
        continue
    take = min(70, len(candidates))
    for i in rng.choice(len(candidates), size=take, replace=False):
        add_entry(candidates[i])

# fill up to target size with random remaining
TARGET_POOL = 1000
if len(pool) < TARGET_POOL:
    remaining = [e for e in entries if e["title"].lower() not in seen]
    extra_take = min(TARGET_POOL - len(pool), len(remaining))
    if extra_take > 0:
        for i in rng.choice(len(remaining), size=extra_take, replace=False):
            add_entry(remaining[i])

print(f"Eval pool size: {len(pool)} anime")

# Test queries
queries = [
    "cute girls doing cute things in high school",
    "dark psychological thriller with mind games",
    "mecha robots fighting in space",
    "romantic comedy with tsundere characters",
    "isekai protagonist transported to fantasy world",
    "sports anime about basketball",
    "slow-burn mystery with plot twists",
    "fantasy adventure with magic and guilds",
    "emotional drama about friendship and loss",
    "high energy shounen battles with power ups",
]

# Encode normalized (cosine = dot product)
query_emb = trained_model.encode(
    queries, normalize_embeddings=True, show_progress_bar=True
)
doc_emb = trained_model.encode(
    [e["text"] for e in pool],
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True
)

sim = query_emb @ doc_emb.T  # cosine similarities

TOP_K = 10
print("\nQuery -> Top matches")
print("=" * 80)
for qi, q in enumerate(queries):
    top_idx = np.argsort(-sim[qi])[:TOP_K]
    print(f"\n{q}")
    for rank, idx in enumerate(top_idx, 1):
        print(f"  {rank:>2}. {pool[idx]['title']} ({sim[qi, idx]:.3f})")


Eval pool size: 1000 anime


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]


Query -> Top matches

cute girls doing cute things in high school
   1. Yuru Yuri, (0.517)
   2. Hourou Musuko (0.507)
   3. Hidamari Sketch x Honeycomb (0.490)
   4. Gochuumon wa Usagi desu ka?? Sing For You (0.482)
   5. Momokuri (0.479)
   6. Watashi ga Motenai no wa Dou Kangaetemo Omaera ga Warui! (0.451)
   7. Uma Musume: Pretty Derby - 1st Anniversary Special Animation (0.445)
   8. Boku no Kanojo ga Majime Sugiru Shoujo Bitch na Ken (0.433)
   9. Isshuukan Friends. Kaori no Nikki (0.430)
  10. Anima Yell! (0.423)

dark psychological thriller with mind games
   1. Tasogare Otome x Amnesia: Taima Otome (0.484)
   2. PSYCHO-PASS 3 (0.468)
   3. Accel World (0.438)
   4. Occultic;Nine (0.428)
   5. ID: INVADED (0.419)
   6. pet (0.417)
   7. Shibou Yuugi de Meshi wo Kuu. (0.412)
   8. Mirai Nikki (0.409)
   9. PSYCHO-PASS Sinners of the System Case 2: First Guardian (0.408)
  10. Satsuriku no Tenshi (ONA) (0.402)

mecha robots fighting in space
   1. Mecha-ude (0.519)
   2. Top wo 

In [16]:
# Side-by-side eval: BASE vs FINE-TUNED on the same large anime pool

import json
import re
import numpy as np
from collections import defaultdict
from sentence_transformers import SentenceTransformer

# Models
base_model = SentenceTransformer(BASE_MODEL, device=str(device))
fine_model = SentenceTransformer(triplet_output_dir, device=str(device))

# Build eval pool from corpus.jsonl
entries = []
with open(DATA_DIR / "corpus.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        x = json.loads(line)
        title = (x.get("title") or "").strip()
        text = (x.get("text") or "").strip()
        if not title or not text:
            continue
        m = re.search(r"^Genres:\s*(.+)$", text, flags=re.MULTILINE)
        genres = [g.strip() for g in m.group(1).split(",")] if m else []
        entries.append({"title": title, "text": text[:1400], "genres": genres})

# Dedupe by title
dedup = {}
for e in entries:
    dedup[e["title"].lower()] = e
entries = list(dedup.values())

# Genre-balanced sampling
rng = np.random.default_rng(42)
genre_targets = [
    "Action", "Adventure", "Comedy", "Romance", "Slice of Life",
    "Psychological", "Thriller", "Mystery", "Sci-Fi", "Mecha",
    "Fantasy", "Supernatural", "Sports", "Drama", "Horror"
]
by_genre = defaultdict(list)
for e in entries:
    for g in e["genres"]:
        by_genre[g].append(e)

pool, seen = [], set()
def add_e(e):
    k = e["title"].lower()
    if k not in seen:
        seen.add(k)
        pool.append(e)

for g in genre_targets:
    c = by_genre.get(g, [])
    if c:
        take = min(70, len(c))
        for i in rng.choice(len(c), size=take, replace=False):
            add_e(c[i])

TARGET_POOL = 1200
if len(pool) < TARGET_POOL:
    rem = [e for e in entries if e["title"].lower() not in seen]
    take = min(TARGET_POOL - len(pool), len(rem))
    if take > 0:
        for i in rng.choice(len(rem), size=take, replace=False):
            add_e(rem[i])

print(f"Eval pool size: {len(pool)}")

queries = [
    "cute girls doing cute things in high school",
    "dark psychological thriller with mind games",
    "mecha robots fighting in space",
    "romantic comedy with tsundere characters",
    "isekai protagonist transported to fantasy world",
    "sports anime about basketball",
]

# Optional weak expected anchors for sanity
expected = {
    "cute girls doing cute things in high school": ["K-On!"],
    "dark psychological thriller with mind games": ["Death Note"],
    "mecha robots fighting in space": ["Evangelion"],
    "romantic comedy with tsundere characters": ["Toradora"],
    "isekai protagonist transported to fantasy world": ["Re:Zero"],
    "sports anime about basketball": ["Slam Dunk"],
}

docs = [e["text"] for e in pool]
titles = [e["title"] for e in pool]

# Encode normalized
q_base = base_model.encode(queries, normalize_embeddings=True, show_progress_bar=True)
d_base = base_model.encode(docs, normalize_embeddings=True, batch_size=64, show_progress_bar=True)

q_fine = fine_model.encode(queries, normalize_embeddings=True, show_progress_bar=True)
d_fine = fine_model.encode(docs, normalize_embeddings=True, batch_size=64, show_progress_bar=True)

sim_base = q_base @ d_base.T
sim_fine = q_fine @ d_fine.T

TOP_K = 10
print("\nBASE vs FINE-TUNED (Top-10)")
print("=" * 100)

base_hits, fine_hits = 0, 0

for i, q in enumerate(queries):
    b_idx = np.argsort(-sim_base[i])[:TOP_K]
    f_idx = np.argsort(-sim_fine[i])[:TOP_K]

    b_titles = [titles[j] for j in b_idx]
    f_titles = [titles[j] for j in f_idx]

    overlap = len(set(b_titles) & set(f_titles)) / TOP_K

    b_top1 = b_titles[0]
    f_top1 = f_titles[0]
    b_s = sim_base[i, b_idx[0]]
    f_s = sim_fine[i, f_idx[0]]

    exp = expected.get(q, [])
    b_hit = any(any(x.lower() in t.lower() for t in b_titles) for x in exp)
    f_hit = any(any(x.lower() in t.lower() for t in f_titles) for x in exp)
    base_hits += int(b_hit)
    fine_hits += int(f_hit)

    print(f"\nQ: {q}")
    print(f"  BASE top1: {b_top1} ({b_s:.3f}) | expected@10: {'Y' if b_hit else 'N'}")
    print(f"  FINE top1: {f_top1} ({f_s:.3f}) | expected@10: {'Y' if f_hit else 'N'}")
    print(f"  Top-10 overlap (Jaccard@10 proxy): {overlap:.2f}")

    print("  BASE top5:")
    for r, j in enumerate(b_idx[:5], 1):
        print(f"    {r}. {titles[j]} ({sim_base[i, j]:.3f})")
    print("  FINE top5:")
    for r, j in enumerate(f_idx[:5], 1):
        print(f"    {r}. {titles[j]} ({sim_fine[i, j]:.3f})")

print("\nSummary")
print(f"  BASE expected-hit@10: {base_hits}/{len(queries)}")
print(f"  FINE expected-hit@10: {fine_hits}/{len(queries)}")


Eval pool size: 1200


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]


BASE vs FINE-TUNED (Top-10)

Q: cute girls doing cute things in high school
  BASE top1: Girlfriend (Kari) (0.528) | expected@10: N
  FINE top1: Girlfriend (Kari) (0.561) | expected@10: N
  Top-10 overlap (Jaccard@10 proxy): 0.40
  BASE top5:
    1. Girlfriend (Kari) (0.528)
    2. Ichigo Mashimaro (0.434)
    3. Battle Girl High School (0.423)
    4. Nichijou no 0-wa (0.413)
    5. Yuru Yuri, (0.381)
  FINE top5:
    1. Girlfriend (Kari) (0.561)
    2. Chuu Bra!! (0.548)
    3. Candy☆Boy (0.537)
    4. Yuru Yuri, (0.517)
    5. Hourou Musuko (0.507)

Q: dark psychological thriller with mind games
  BASE top1: ID: INVADED (0.550) | expected@10: N
  FINE top1: Tasogare Otome x Amnesia: Taima Otome (0.484) | expected@10: N
  Top-10 overlap (Jaccard@10 proxy): 0.40
  BASE top5:
    1. ID: INVADED (0.550)
    2. Accel World (0.494)
    3. pet (0.481)
    4. Tasogare Otome x Amnesia: Taima Otome (0.469)
    5. Kaiba (0.467)
  FINE top5:
    1. Tasogare Otome x Amnesia: Taima Otome (0.484)


In [17]:
print(f"\n✓ Model saved to: {triplet_output_dir}")
print(f"  Embedding dimension: {trained_model.get_sentence_embedding_dimension()}")
print(f"  Model size: ~{sum(p.numel() * p.element_size() for p in trained_model[0].auto_model.parameters()) / 1e6:.0f} MB")
print(f"\nProceed to 04_cf_training.ipynb or 05_export.ipynb")


✓ Model saved to: models\anime_semantic
  Embedding dimension: 384
  Model size: ~91 MB

Proceed to 04_cf_training.ipynb or 05_export.ipynb
